# 6주차. FastAPI 사전 지식과 SQLite CRUD

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 데코레이터와 레지스트리
FastAPI의 라우터 데코레이터는 함수와 URL을 연결합니다. 아래 예제는 같은 원리를 작은 레지스트리로 표현합니다.

In [1]:
routes = {}

def route(path):
    def decorator(func):
        routes[path] = func
        return func
    return decorator

@route("/health")
def health():
    return {"status": "ok"}

assert routes["/health"]() == {"status": "ok"}
routes

{'/health': <function __main__.health()>}

## SQLite CRUD
CRUD는 생성, 조회, 수정, 삭제입니다. 입력 검증을 먼저 수행하고 데이터베이스에 반영합니다.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE customers (id TEXT PRIMARY KEY, name TEXT, age INTEGER, grade TEXT)")

def validate_customer(row):
    if not row["id"] or not row["name"]:
        raise ValueError("id and name are required")
    if not 0 < row["age"] < 120:
        raise ValueError("age out of range")
    return row

def create_customer(row):
    row = validate_customer(row)
    conn.execute("INSERT INTO customers VALUES (?, ?, ?, ?)", (row["id"], row["name"], row["age"], row["grade"]))
    conn.commit()

def read_customers():
    return pd.read_sql_query("SELECT * FROM customers", conn)

create_customer({"id": "C001", "name": "Sungmin", "age": 22, "grade": "gold"})
create_customer({"id": "C002", "name": "Min", "age": 21, "grade": "silver"})
df = read_customers()

assert len(df) == 2
display(df)

,id,name,age,grade
0,C001,Sungmin,22,gold
1,C002,Min,21,silver
